In [1]:
# ============================================================
# FUNCTION 4 — WEEK 6 CLEAN REBUILD
# ============================================================

import numpy as np
import matplotlib.pyplot as plt

from scipy.stats import norm
from scipy.spatial import cKDTree

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import (
    ConstantKernel as C,
    Matern,
    WhiteKernel
)

# ------------------------------------------------------------
# 1. Load original Function 4 data
# ------------------------------------------------------------

X = np.load("function4/initial_inputs.npy")
Y = np.load("function4/initial_outputs.npy").reshape(-1)

# ------------------------------------------------------------
# 2. Add Weeks 1–5 exactly once
# ------------------------------------------------------------

weekly_X = np.array([
    [0.370368, 0.421656, 0.371701, 0.447899],  # Week 1
    [0.413742, 0.470182, 0.290309, 0.439853],  # Week 2
    [0.379028, 0.410913, 0.447054, 0.442255],  # Week 3
    [0.339565, 0.421924, 0.397318, 0.438082],  # Week 4
    [0.357812, 0.420904, 0.424244, 0.430762]   # Week 5
])

weekly_Y = np.array([
     0.17319018408919762,
    -1.7902558622959037,
    -0.06479142792096892,
     0.2806521851858581,
     0.6010598713446957
])

X = np.vstack([
    X,
    weekly_X
])

Y = np.concatenate([
    Y,
    weekly_Y
])

best_index = np.argmax(Y)
best_x = X[best_index]
best_y = Y[best_index]

print("Week 6 X shape:", X.shape)
print("Week 6 Y shape:", Y.shape)

print("\nBest observed input:")
print(best_x)

print("\nBest observed output:")
print(best_y)

print("\nWeek 5 input:")
print(weekly_X[-1])

print("Week 5 output:")
print(weekly_Y[-1])

Week 6 X shape: (35, 4)
Week 6 Y shape: (35,)

Best observed input:
[0.357812 0.420904 0.424244 0.430762]

Best observed output:
0.6010598713446957

Week 5 input:
[0.357812 0.420904 0.424244 0.430762]
Week 5 output:
0.6010598713446957


In [2]:
# ============================================================
# 3. FIT ARD GAUSSIAN PROCESS
# ============================================================

kernel = (
    C(
        1.0,
        (1e-3, 1e3)
    )
    *
    Matern(
        length_scale=[
            0.15,
            0.15,
            0.12,
            0.15
        ],
        length_scale_bounds=(
            0.005,
            3.0
        ),
        nu=2.5
    )
    +
    WhiteKernel(
        noise_level=1e-6,
        noise_level_bounds=(
            1e-10,
            1e-2
        )
    )
)

gp = GaussianProcessRegressor(
    kernel=kernel,
    normalize_y=True,
    n_restarts_optimizer=20,
    random_state=64
)

gp.fit(
    X,
    Y
)

print("Fitted kernel:")
print(gp.kernel_)

# ------------------------------------------------------------
# Input sensitivity
# ------------------------------------------------------------

lengthscales = np.asarray(
    gp.kernel_.k1.k2.length_scale
)

importance = 1 / lengthscales
importance = importance / importance.sum()

print("\nFitted lengthscales:")
print(lengthscales)

print("\nNormalised input influence:")

for i, value in enumerate(
    importance,
    start=1
):
    print(
        f"x{i}: {value:.4f}"
    )

Fitted kernel:
2.59**2 * Matern(length_scale=[1.63, 1.33, 1.33, 1.4], nu=2.5) + WhiteKernel(noise_level=0.000826)

Fitted lengthscales:
[1.63207681 1.33329083 1.32838407 1.40291101]

Normalised input influence:
x1: 0.2166
x2: 0.2652
x3: 0.2662
x4: 0.2520


In [3]:
# ============================================================
# 4. GENERATE WEEK 6 CANDIDATES
# ============================================================

rng = np.random.default_rng(64)

week5_point = np.array([
    0.357812,
    0.420904,
    0.424244,
    0.430762
])

week4_point = np.array([
    0.339565,
    0.421924,
    0.397318,
    0.438082
])

week1_point = np.array([
    0.370368,
    0.421656,
    0.371701,
    0.447899
])

# ------------------------------------------------------------
# A. Very tight exploitation around Week 5
# ------------------------------------------------------------

very_local = rng.normal(
    loc=week5_point,
    scale=[
        0.004,
        0.004,
        0.004,
        0.004
    ],
    size=(20_000, 4)
)

# ------------------------------------------------------------
# B. Local search around Week 5
# ------------------------------------------------------------

local = rng.normal(
    loc=week5_point,
    scale=[
        0.012,
        0.012,
        0.012,
        0.012
    ],
    size=(20_000, 4)
)

# ------------------------------------------------------------
# C. Slightly wider local search
# ------------------------------------------------------------

wider_local = rng.normal(
    loc=week5_point,
    scale=[
        0.030,
        0.025,
        0.025,
        0.025
    ],
    size=(10_000, 4)
)

# ------------------------------------------------------------
# D. Blend Week 4 and Week 5
#
# Both performed well, with Week 5 much stronger.
# ------------------------------------------------------------

weights = rng.uniform(
    0,
    1,
    size=(10_000, 1)
)

blend_45 = (
    weights * week5_point
    +
    (1 - weights) * week4_point
)

blend_45 += rng.normal(
    0,
    [
        0.006,
        0.006,
        0.006,
        0.006
    ],
    size=(10_000, 4)
)

# ------------------------------------------------------------
# E. Small search around Week 1 / Week 5 region
# ------------------------------------------------------------

weights_15 = rng.uniform(
    0.65,
    1.0,
    size=(5_000, 1)
)

blend_15 = (
    weights_15 * week5_point
    +
    (1 - weights_15) * week1_point
)

blend_15 += rng.normal(
    0,
    [
        0.006,
        0.006,
        0.006,
        0.006
    ],
    size=(5_000, 4)
)

candidates = np.vstack([
    very_local,
    local,
    wider_local,
    blend_45,
    blend_15
])

candidates = np.clip(
    candidates,
    0,
    1
)

print(
    "Candidates before filtering:",
    len(candidates)
)

Candidates before filtering: 65000


In [4]:
# ============================================================
# 5. REMOVE NEAR-DUPLICATES
# ============================================================

tree = cKDTree(X)

minimum_distance, _ = tree.query(
    candidates,
    k=1
)

keep = minimum_distance > 0.0025

candidates = candidates[
    keep
]

print(
    "Candidates after filtering:",
    len(candidates)
)

Candidates after filtering: 64651


In [5]:
# ============================================================
# 6. GP PREDICTIONS
# ============================================================

mean, std = gp.predict(
    candidates,
    return_std=True
)

# ------------------------------------------------------------
# Expected Improvement
# ------------------------------------------------------------

xi = 0.001

improvement = (
    mean
    - best_y
    - xi
)

with np.errstate(
    divide="ignore",
    invalid="ignore"
):

    z = improvement / std

    ei = (
        improvement * norm.cdf(z)
        +
        std * norm.pdf(z)
    )

ei[std < 1e-12] = 0

# ------------------------------------------------------------
# Small UCB component
# ------------------------------------------------------------

kappa = 0.30

ucb = (
    mean
    + kappa * std
)

In [6]:
# ============================================================
# 7. FILTER OUT WEAK PREDICTED CANDIDATES
# ============================================================

mean_filter = (
    mean >= best_y - 0.10
)

if np.sum(mean_filter) < 100:
    mean_filter = (
        mean >= np.percentile(
            mean,
            90
        )
    )

filtered_candidates = candidates[
    mean_filter
]

filtered_mean = mean[
    mean_filter
]

filtered_std = std[
    mean_filter
]

filtered_ei = ei[
    mean_filter
]

filtered_ucb = ucb[
    mean_filter
]

print(
    "Candidates passing mean filter:",
    len(filtered_candidates)
)

Candidates passing mean filter: 6466


In [7]:
# ============================================================
# 8. ACQUISITION SCORE
# ============================================================

ei_norm = (
    filtered_ei
    - filtered_ei.min()
) / (
    np.ptp(filtered_ei)
    + 1e-12
)

ucb_norm = (
    filtered_ucb
    - filtered_ucb.min()
) / (
    np.ptp(filtered_ucb)
    + 1e-12
)

acquisition = (
    0.90 * ei_norm
    +
    0.10 * ucb_norm
)

chosen_index = np.argmax(
    acquisition
)

week6_query = filtered_candidates[
    chosen_index
]

In [8]:
# ============================================================
# 9. FUNCTION 4 — WEEK 6 PORTAL OUTPUT
# ============================================================

print("\nSuggested Week 6 query:")
print(week6_query)

print("\nPortal format:")
print(
    "-".join(
        f"{value:.6f}"
        for value in week6_query
    )
)

print(
    "\nPredicted mean:",
    filtered_mean[chosen_index]
)

print(
    "Predicted uncertainty:",
    filtered_std[chosen_index]
)

print(
    "Expected Improvement:",
    filtered_ei[chosen_index]
)

print(
    "UCB:",
    filtered_ucb[chosen_index]
)

print(
    "Distance from Week 5:",
    np.linalg.norm(
        week6_query
        - week5_point
    )
)

print(
    "Distance from Week 4:",
    np.linalg.norm(
        week6_query
        - week4_point
    )
)


Suggested Week 6 query:
[0.35400783 0.42585252 0.42877254 0.42774534]

Portal format:
0.354008-0.425853-0.428773-0.427745

Predicted mean: 0.30137694186284314
Predicted uncertainty: 0.29898635528798856
Expected Improvement: 0.0246421810126441
UCB: 0.39107284844923973
Distance from Week 5: 0.008280544911793105
Distance from Week 4: 0.036335430145320556
